# Implicit time stepping: sensitivity of landscape evolution to $\Delta t$

This example explores how the choice of simulation time step $\Delta t$ affects a `gospl` detachment-limited landscape that is uplifted to steady state. Four otherwise identical simulations are run with $\Delta t = 500$, $1250$, $2500$ and $5000$ yr (input files `input-500.yml` to `input-5k.yml`), each solving the stream-power law $E = K\,A^{m}\,S^{n}$ with `gospl`'s implicit, unconditionally stable flow-accumulation and erosion solvers.

Because the scheme is implicit, large $\Delta t$ values do not blow up the solution (unlike an explicit Courant-limited scheme); instead the trade-off appears as accuracy and runtime differences. Here we post-process the `gospl` outputs onto a regular UTM grid, export them to netCDF, and compare the final topographies and the mean-elevation histories to quantify that trade-off.

Import required Python packages for this notebook.


In [ ]:
import importlib, subprocess, sys

def ensure_installed(package):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

ensure_installed("pysheds")

Import required Python packages for this notebook.


In [ ]:
import os
import numpy as np
import xarray as xr

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from scripts import mapOutputs as mout

import matplotlib.pyplot as plt
%matplotlib inline

**Shared simulation parameters (from the `input-*.yml` files)**

All four runs share the configuration below; only the `time:dt` entry differs (500, 1250, 2500 or 5000 yr).

| Parameter | Symbol | Value | Physical meaning |
|---|---|---|---|
| `time:end` | $t_{end}$ | $10^{5}$ yr | total simulated time (long enough to reach steady state) |
| `time:dt` | $\Delta t$ | 500–5000 yr | solver time step, the variable studied here |
| `time:tout` | — | 2500 yr | output interval written to disk |
| `spl:K` | $K$ | $2\times10^{-4}$ | bedrock erodibility in $E = K\,A^{m}\,S^{n}$ |
| `spl:m` | $m$ | 0.5 | drainage-area exponent (with $n=1$) |
| `diffusion:hillslopeKa` | $\kappa$ | 0 | hillslope diffusion off (purely fluvial) |
| `domain:flowdir` | — | 1 | single-flow-direction routing |
| `domain:bc` | — | `0000` | fixed (open) domain boundaries |
| `domain:nodep` | — | True | detachment-limited, no deposition |
| `climate:uniform` | $P$ | 1 m/yr | uniform precipitation |
| `tectonics:upsub` | $U$ | 0–5 mm/yr | linear west-to-east uplift gradient |

**This example illustrates the effect of increasing time step length on the resulting landscape evolution.**

#### Mesh creation

In this example, we use the same mesh as the one defined in the `generate_mesh` folder. The initial surface consists of a flat triangulated squared grid (with noise) with 100 km sides and 200 m resolution

It defines a flat elevation at 200 m (with noise) and a simple tectonic uplift (with a linear slope ranging to 5 mm/yr). 

The goSPL input file (`gospl_mesh.npz`) has been copied in the `data` folder. Below we provide the code needed to regenerate it if necessary.

### Extracting a single output to a regular grid

We instantiate `mapOutputs` for the 500 yr run, asking it to also load the `uplift` field (`flex=False` as no flexural isostasy is computed here). `buildUTMmesh` interpolates the unstructured `gospl` variables onto a 200 m regular UTM mesh using inverse-distance weighting (`nghb=4` neighbours, light `smth=0.5` smoothing), and `exportNetCDF` writes the result. Here `step=10` selects the output time slice ($10 \times t_{out} = 25$ ky).

In [ ]:
make_mesh = False
if make_mesh:
    import shutil
    import meshio
    import uxarray as uxr
    from scripts import umeshFcts as ufcts
    from gospl.mesher.meshfunc import VoroBuild

    output_path = "slope_tec" 
    shutil.rmtree(output_path, ignore_errors=True)
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    dx = 200 # desired resolution
    nx = 501 # desired number of nodes along the x-axis
    ny = 501 # desired number of nodes along the y-axis

    tmin = 0.
    tmax = 0.005

    xcoords = np.arange(nx)*float(dx) 
    ycoords = np.arange(ny)*float(dx) 
    tecx = np.interp(xcoords, [xcoords[0],xcoords[-1]], [tmin,tmax])
    tec = np.broadcast_to(tecx, (nx, nx))[:ny,:]


    noise = np.random.normal(0, 0.05, tec.shape)
    elev = noise+100.

    ds = xr.Dataset({
        'elev': xr.DataArray(
                    data   = elev,
                    dims   = ['y','x'],
                    coords = {'x': xcoords,'y': ycoords},
                    ),
        'tec': xr.DataArray(
                    data   = tec,
                    dims   = ['y','x'],
                    coords = {'x': xcoords,'y': ycoords},
                    )
            }
        )
    ds['cellwidth'] = (['y','x'],dx*np.ones( (ny, nx)))
        
    # Build your planar mesh
    ufcts.planarMesh(ds,output_path,fvtk='planar.vtk',fumpas=True,voro=True)

    # Loading the UGRID file
    ufile = output_path+'/base2D.nc'
    var_name = 'data'
    mapds = xr.open_dataset(ufile) 

    # Perform the interpolation (bilinear) 
    ufcts.inter2UGRID(ds[['elev','tec']],mapds,output_path,var_name,type='face',latlon=False)
    # ufcts.inter2UGRID(ds[['t']],mapds,output_path,var_name,type='face',latlon=False)
    data_ds = xr.open_dataset(output_path + '/' + var_name + '.nc')

    # Extract nodes and faces information
    n_nodes = mapds.dims['nCells']
    ucoords = np.zeros((n_nodes, 3))
    ucoords[:, 0] = mapds['xCell'].values
    ucoords[:, 1] = mapds['yCell'].values
    ucoords[:, 2] = mapds['zCell'].values
    ufaces = mapds['cellsOnVertex'].values - 1 
    print(f"Number of nodes: {len(ucoords)} | Number of faces: {len(ufaces)}")

    # Get information about your mesh:
    dcEdge = mapds['dcEdge'].values  # in metres
    edge_min  = np.round(dcEdge.min()  / 1000., 2)
    edge_max  = np.round(dcEdge.max()  / 1000., 2)
    edge_mean = np.round(dcEdge.mean() / 1000., 2)
    print("edge range (km): min ",edge_min," | max ",edge_max," | mean ",edge_mean)

    mesh = meshio.read(output_path+'/planar.vtk')
    vertex = mesh.points
    cells = mesh.cells_dict['triangle']
    Umesh = VoroBuild()
    Umesh.initVoronoi(vertex, cells)
    Uarea = Umesh.control_volumes
    print('Cell area (km2): ',Uarea.min()*1.e-6,Uarea.max()*1.e-6)

    meshname = "data/gospl_mesh"
    np.savez_compressed(meshname, v=vertex, c=cells, 
                    z=data_ds.elev.data, t=data_ds.tec.data
                    )


You will find a series of goSPL input files:
- input-500.yml
- input-1.25k.yml
- input-2.5k.yml
- input-5k.yml

These files use the same elevation and forcing conditions, the only difference being the time step used to solve the stream power equation (ranging from 500 yrs to 5,000 years).

The surface is exposed to an uniform precipitation regime of 1 m/yr and is uplifted linearly from its fixed western side to the eastern one that experiences an uplift of 5 mm/yr. 

The value of the bedrock erodibility parameter K is set to 2e-4 in order to reach steady state during the simulated 1.e5 years. The model is purely erosional and therefore marine sedimentation is not considered. In addition, hillslope processes are also turned off, meaning that this example only relies on the implicit parallel flow discharge (using a single flow direction approach) and erosion equations.

In both cases the implicit schemas converge for the chosen solver and preconditioner (i.e. Richardson with block Jacobian).